<a href="https://colab.research.google.com/github/RajendharAre/Innomatics_Research_Labs_IN226052302/blob/main/GenAI_NLP_Task5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import transformers
from datasets import load_dataset, Dataset, DatasetDict, Features, Sequence, ClassLabel, Value

print(f"Transformers version: {transformers.__version__}")

# Upgrade the datasets library to potentially resolve loading issues
!pip install --upgrade datasets

Transformers version: 5.0.0


### Task 1: Dataset Selection

**Dataset Name**: Universal Dependencies

In [4]:
def read_conll_data(file_path):
    all_tokens = []
    all_upos_tags_str = []
    all_xpos_tags_str = []

    current_tokens = []
    current_upos_str = []
    current_xpos_str = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line.startswith('#'):  # Skip comments
                continue
            if line: # If line is not empty
                parts = line.split('\t')
                # The file format is TOKEN, UPOS, XPOS
                if len(parts) == 3:
                    token = parts[0]
                    upos = parts[1]
                    xpos = parts[2]

                    current_tokens.append(token)
                    current_upos_str.append(upos)
                    current_xpos_str.append(xpos)
                else:
                    # If a line doesn't have 3 parts, it likely indicates a sentence boundary or malformed data
                    # Treat it as a sentence boundary if there are accumulated tokens
                    if current_tokens:
                        all_tokens.append(current_tokens)
                        all_upos_tags_str.append(current_upos_str)
                        all_xpos_tags_str.append(current_xpos_str)
                        current_tokens = []
                        current_upos_str = []
                        current_xpos_str = []
            else:
                # Empty line signifies a sentence boundary
                if current_tokens:
                    all_tokens.append(current_tokens)
                    all_upos_tags_str.append(current_upos_str)
                    all_xpos_tags_str.append(current_xpos_str)
                    current_tokens = []
                    current_upos_str = []
                    current_xpos_str = []
    # Add the last sentence if the file doesn't end with an empty line
    if current_tokens:
        all_tokens.append(current_tokens)
        all_upos_tags_str.append(current_upos_str)
        all_xpos_tags_str.append(current_xpos_str)

    return {"tokens": all_tokens, "upos": all_upos_tags_str, "xpos": all_xpos_tags_str}

# Load the data from the provided local text files
train_parsed_data = read_conll_data('/content/en-ud-tag.v2.train.txt')
test_parsed_data = read_conll_data('/content/en-ud-tag.v2.test.txt')

print(f"Train parsed data sentences: {len(train_parsed_data['tokens'])}")
print(f"Test parsed data sentences: {len(test_parsed_data['tokens'])}")

# Collect all unique UPOS and XPOS tags from the training set to define consistent ClassLabels
all_upos_tags = sorted(list(set(tag for sentence_tags in train_parsed_data['upos'] for tag in sentence_tags)))
all_xpos_tags = sorted(list(set(tag for sentence_tags in train_parsed_data['xpos'] for tag in sentence_tags)))

# Define the features schema for the Dataset
features = Features({
    "tokens": Sequence(Value('string')),
    "upos": Sequence(ClassLabel(names=all_upos_tags)),
    "xpos": Sequence(ClassLabel(names=all_xpos_tags)),
})

# Create Dataset objects with the defined features, which will map string tags to integer IDs
train_dataset = Dataset.from_dict(train_parsed_data, features=features)
test_dataset = Dataset.from_dict(test_parsed_data, features=features)

# Create a DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

print("Dataset loaded successfully from local files:")
print(dataset)

Train parsed data sentences: 12542
Test parsed data sentences: 2077
Dataset loaded successfully from local files:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'upos', 'xpos'],
        num_rows: 12542
    })
    test: Dataset({
        features: ['tokens', 'upos', 'xpos'],
        num_rows: 2077
    })
})


In [5]:
# Inspect the first 20 lines of the training data file to understand its format
print("First 20 lines of /content/en-ud-tag.v2.train.txt:")
with open('/content/en-ud-tag.v2.train.txt', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 20: # Displaying first 20 lines
            break
        print(line.strip())


First 20 lines of /content/en-ud-tag.v2.train.txt:
Al	PROPN	NNP
-	PUNCT	HYPH
Zaman	PROPN	NNP
:	PUNCT	:
American	ADJ	JJ
forces	NOUN	NNS
killed	VERB	VBD
Shaikh	PROPN	NNP
Abdullah	PROPN	NNP
al	PROPN	NNP
-	PUNCT	HYPH
Ani	PROPN	NNP
,	PUNCT	,
the	DET	DT
preacher	NOUN	NN
at	ADP	IN
the	DET	DT
mosque	NOUN	NN
in	ADP	IN
the	DET	DT


Let's inspect the structure of the dataset and look at an example to understand its features, especially the POS and chunk tags.

In [6]:
# Display an example from the training set
example = dataset["train"][0]
print("\nFirst example from the training set:")
print(example)

# Get the feature names for Universal POS and Language-specific POS tags
upos_tag_names = dataset["train"].features["upos"].feature.names
xpos_tag_names = dataset["train"].features["xpos"].feature.names

print("\nUniversal POS Tag Names (UPOS):")
print(upos_tag_names)

print("\nLanguage-specific POS Tag Names (XPOS):")
print(xpos_tag_names)

# Map numerical tags to their string representations for the example
print("\nExample with mapped tags:")
print(f"Tokens: {example['tokens']}")
print(f"UPOS Tags: {[upos_tag_names[tag_id] for tag_id in example['upos']]}")
print(f"XPOS Tags: {[xpos_tag_names[tag_id] for tag_id in example['xpos']]}")


First example from the training set:
{'tokens': ['Al', '-', 'Zaman', ':', 'American', 'forces', 'killed', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.'], 'upos': [11, 12, 11, 12, 0, 7, 15, 11, 11, 11, 12, 11, 12, 5, 7, 1, 5, 7, 1, 5, 7, 1, 11, 12, 1, 5, 0, 7, 12], 'xpos': [24, 15, 24, 6, 17, 26, 39, 24, 24, 24, 15, 24, 2, 11, 23, 16, 11, 23, 16, 11, 23, 16, 24, 2, 16, 11, 17, 23, 5]}

Universal POS Tag Names (UPOS):
['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']

Language-specific POS Tag Names (XPOS):
['$', "''", ',', '-LRB-', '-RRB-', '.', ':', 'ADD', 'AFX', 'CC', 'CD', 'DT', 'EX', 'FW', 'GW', 'HYPH', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NFP', 'NN', 'NNP', 'NNPS', 'NNS', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VB

### Task 2: Data Processing
In this step, we tokenize the text and align the labels (UPOS) with the tokens, handling subwords by assigning `-100` to non-initial subword tokens.

In [7]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples["upos"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens like [CLS] or [SEP] get -100
            if word_idx is None:
                label_ids.append(-100)
            # We only label the first token of a given word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # Subsequent subwords also get -100
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [8]:
from transformers import AutoTokenizer

model_checkpoint = "prajjwal1/bert-tiny"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

print(f"Tokenizer loaded: {model_checkpoint}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Tokenizer loaded: prajjwal1/bert-tiny


In [9]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

print("Data processing complete. Structure of the tokenized dataset:")
print(tokenized_dataset)

Map:   0%|          | 0/12542 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

Data processing complete. Structure of the tokenized dataset:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'upos', 'xpos', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 12542
    })
    test: Dataset({
        features: ['tokens', 'upos', 'xpos', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2077
    })
})


In [10]:
# Inspect the first processed example
first_example = tokenized_dataset['train'][0]
print("Sample processed labels:", first_example['labels'])
print("Sample input IDs:", first_example['input_ids'])
print("Tokens decoded:", tokenizer.convert_ids_to_tokens(first_example['input_ids']))

Sample processed labels: [-100, 11, 12, 11, 12, 0, 7, 15, 11, 11, 11, 12, 11, 12, 5, 7, 1, 5, 7, 1, 5, 7, 1, 11, 12, 1, 5, 0, 7, 12, -100]
Sample input IDs: [101, 100, 1011, 100, 1024, 100, 2749, 2730, 100, 100, 2632, 1011, 100, 1010, 1996, 14512, 2012, 1996, 8806, 1999, 1996, 2237, 1997, 100, 1010, 2379, 1996, 100, 3675, 1012, 102]
Tokens decoded: ['[CLS]', '[UNK]', '-', '[UNK]', ':', '[UNK]', 'forces', 'killed', '[UNK]', '[UNK]', 'al', '-', '[UNK]', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', '[UNK]', ',', 'near', 'the', '[UNK]', 'border', '.', '[SEP]']


## Model Training

In [11]:
from transformers import AutoModelForTokenClassification

# Get the number of labels from the UPOS tag names
num_labels = len(upos_tag_names)

id2label = {i: label for i, label in enumerate(upos_tag_names)}
label2id = {label: i for i, label in enumerate(upos_tag_names)}

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

print(f"Model loaded: {model_checkpoint} with {num_labels} labels.")

pytorch_model.bin:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/37 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSI

Model loaded: prajjwal1/bert-tiny with 17 labels.


## Training

In [12]:
!pip install --quiet evaluate seqeval

In [13]:
import evaluate
import numpy as np
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

# Define training arguments
# evaluation_strategy is deprecated, use eval_strategy instead
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=32, # Increased batch size
    per_device_eval_batch_size=32,  # Increased batch size
    num_train_epochs=1, # Reduced for faster training
    weight_decay=0.01,
    eval_strategy="epoch", # Changed from evaluation_strategy
    save_strategy="epoch", # Changed from save_strategy
    load_best_model_at_end=True,
    logging_dir='./logs',
    logging_steps=100,
)

# Data collator to handle padding
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Metric definition
metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (where label is -100)
    true_labels = [[id2label[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer initialized.")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Trainer initialized.


In [14]:
print("Starting model training...")
trainer.train()
print("Model training complete.")

Starting model training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,2.080441,1.902023,0.553436,0.478435,0.513210,0.590219


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PRON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: SCONJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:17

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.encoder.layer.0.output.LayerNorm.beta', 'bert.encoder.layer.0.output.LayerNorm.gamma', 'bert.encoder.layer.1.attention.output.LayerNorm.beta', 'bert.encoder.layer.1.attention.output.LayerNorm.gamma', 'bert.en

Model training complete.


## Evaluation

In [15]:
print("Starting model evaluation...")
results = trainer.evaluate()
print("Evaluation results:")
print(results)

Starting model evaluation...


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PRON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: SCONJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:17

Evaluation results:
{'eval_loss': 1.902023196220398, 'eval_precision': 0.5534358769476628, 'eval_recall': 0.4784354358243751, 'eval_f1': 0.5132099937481187, 'eval_accuracy': 0.5902188209972498, 'eval_runtime': 4.1885, 'eval_samples_per_second': 495.883, 'eval_steps_per_second': 15.519, 'epoch': 1.0}


## Inference

In [16]:
from transformers import pipeline

# Create a pipeline for token classification
classifier = pipeline("token-classification", model=model, tokenizer=tokenizer)

# Example sentence for inference
example_text = "Google Colaboratory is a free cloud-based Jupyter notebook environment."
print(f"Example text for inference: {example_text}")

# Perform inference
preds = classifier(example_text)

print("Inference results:")
for pred in preds:
    print(f"Token: {pred['word']}, Predicted UPOS: {pred['entity']}")

Example text for inference: Google Colaboratory is a free cloud-based Jupyter notebook environment.
Inference results:
Token: Google, Predicted UPOS: PROPN
Token: Colaboratory, Predicted UPOS: PROPN
Token: is, Predicted UPOS: VERB
Token: a, Predicted UPOS: DET
Token: free, Predicted UPOS: NOUN
Token: cloud, Predicted UPOS: NOUN
Token: -, Predicted UPOS: PUNCT
Token: based, Predicted UPOS: NOUN
Token: Jupyter, Predicted UPOS: PROPN
Token: notebook, Predicted UPOS: NOUN
Token: environment, Predicted UPOS: NOUN
Token: ., Predicted UPOS: PUNCT


## Comparison

In [18]:
print("Available Universal POS Tags (UPOS):", upos_tag_names)
print("Available Language-specific POS Tags (XPOS):", xpos_tag_names)

# Get an example from the test set
example_test = dataset["test"][0]

# Perform inference on the original tokens from the test set example
example_tokens = example_test['tokens']
predicted_tags_inference = classifier(example_tokens)

# Extract predicted UPOS tags from the inference results
# The classifier returns a list of lists of dictionaries for a list of tokens.
# Each inner list contains one dictionary corresponding to the token.
predicted_upos = [pred_list[0]['entity'] for pred_list in predicted_tags_inference if pred_list[0]['word'] != '[CLS]' and pred_list[0]['word'] != '[SEP]']

# Map original UPOS tags from numerical ID to string
original_upos = [upos_tag_names[tag_id] for tag_id in example_test['upos']]

print("\nComparison for a test set example:")
print(f"Tokens: {example_test['tokens']}")
print(f"Original UPOS Tags: {original_upos}")
print(f"Predicted UPOS Tags: {predicted_upos}")

Available Universal POS Tags (UPOS): ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']
Available Language-specific POS Tags (XPOS): ['$', "''", ',', '-LRB-', '-RRB-', '.', ':', 'ADD', 'AFX', 'CC', 'CD', 'DT', 'EX', 'FW', 'GW', 'HYPH', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NFP', 'NN', 'NNP', 'NNPS', 'NNS', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB', 'XX', '``']

Comparison for a test set example:
Tokens: ['What', 'if', 'Google', 'Morphed', 'Into', 'GoogleOS', '?']
Original UPOS Tags: ['PRON', 'SCONJ', 'PROPN', 'VERB', 'ADP', 'PROPN', 'PUNCT']
Predicted UPOS Tags: ['PROPN', 'NOUN', 'PROPN', 'PROPN', 'PROPN', 'PROPN', 'PUNCT']
